<a href="https://colab.research.google.com/github/jbloombe/Project0/blob/main/We_Love_Sabermetrics_value_projections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install pybaseball and clean names

In [ ]:
! pip install pybaseball MLB-StatsAPI -q

import os
import pandas as pd
import unicodedata
import numpy as np
from pybaseball import batting_stats, pitching_stats
import requests

def cleanname(name):
    # Lowercase, strip whitespace, remove accents
    name = str(name).strip().lower()
    return ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    )

def pull_multi_year(stat_func, years, col, id_col='IDfg', qual=None):
    dfs = []
    for yr in years:
        # Only pass qual if explicitly set — bref functions don't support it
        if qual is not None:
            df = stat_func(yr, qual=qual)[[id_col, col]]
        else:
            df = stat_func(yr)[[id_col, col]]
        df = df.rename(columns={col: f"{col}_{str(yr)[-2:]}"})
        dfs.append(df)
    result = dfs[0]
    for df in dfs[1:]:
        result = result.merge(df, on=id_col, how='outer')
    return result

def pull_savant_multi_year(savant_func, years, cols, id_col='player_id'):
    # Pull Savant percentile ranks across multiple years, concat, and average
    frames = [savant_func(yr) for yr in years]
    return pd.concat(frames).groupby(id_col)[cols].mean().reset_index()


def mlb_batting(year, qual=1):
    url = (
        f"https://statsapi.mlb.com/api/v1/stats?"
        f"stats=season&group=hitting&season={year}"
        f"&limit=2000&offset=0&sportId=1"
    )
    resp = requests.get(url, timeout=20).json()
    rows = []
    for split in resp['stats'][0]['splits']:
        p = split['player']
        s = split['stat']
        rows.append({'playerid': p['id'], 'G': s.get('gamesPlayed', 0)})
    return pd.DataFrame(rows)

def mlb_pitching(year, qual=1):
    url = (
        f"https://statsapi.mlb.com/api/v1/stats?"
        f"stats=season&group=pitching&season={year}"
        f"&limit=2000&offset=0&sportId=1"
    )
    resp = requests.get(url, timeout=20).json()
    rows = []
    for split in resp['stats'][0]['splits']:
        p = split['player']
        s = split['stat']
        rows.append({'playerid': p['id'], 'IP': float(s.get('inningsPitched', 0))})
    return pd.DataFrame(rows)

SAVE_PATH = '/content/drive/MyDrive/WeLoveSabermetrics/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"✅ Drive mounted. Saving to: {SAVE_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.9 MB/s eta 0:00:00
✅ Drive mounted. Saving to: /content/drive/MyDrive/WeLoveSabermetrics/


# Pull Fangraphs hitters and pitchers

In [ ]:
# --- HITTERS ---
hit_url = (
    "https://www.fangraphs.com/api/projections?"
    "type=steamer&stats=bat&pos=all&team=0&players=0&lg=all"
)
hitters = pd.read_json(hit_url)

# --- PITCHERS ---
pit_url = (
    "https://www.fangraphs.com/api/projections?"
    "type=steamer&stats=pit&pos=all&team=0&players=0&lg=all"
)
pitchers = pd.read_json(pit_url)

print(hitters.columns.tolist())   # inspect available columns first
print(pitchers.columns.tolist())


['Team', 'ShortName', 'G', 'AB', 'PA', 'H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SF', 'SH', 'GDP', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS', 'wOBA', 'BB%', 'K%', 'BB/K', 'ISO', 'Spd', 'BABIP', 'UBR', 'GDPRuns', 'wRC', 'wRAA', 'UZR', 'wBsR', 'BaseRunning', 'WAR', 'Off', 'Def', 'wRC+', 'FPTS', 'FPTS_G', 'SPTS', 'SPTS_G', 'woba_sd', 'truetalent_sd', 'woba_sd_book', 'woba_se', 'total_se', 'q10', 'q20', 'q30', 'q40', 'q50', 'q60', 'q70', 'q80', 'q90', 'tt_q10', 'tt_q20', 'tt_q30', 'tt_q40', 'tt_q50', 'tt_q60', 'tt_q70', 'tt_q80', 'tt_q90', 'ADP', 'Pos', 'minpos', 'UPURL', 'teamid', 'League', 'PlayerName', 'xMLBAMID', 'playerids', 'playerid']
['Team', 'ShortName', 'W', 'L', 'GS', 'G', 'SV', 'HLD', 'BS', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'SO', 'BB', 'IBB', 'HBP', 'ERA', 'WHIP', 'K/9', 'BB/9', 'K/BB', 'HR/9', 'K%', 'BB%', 'K-BB%', 'GB%', 'AVG', 'BABIP', 'LOB%', 'FIP', 'FPTS', 'FPTS_IP', 'SPTS', 'SPTS_IP', 'WAR', 'RA9-WAR', 'QS', 'ra_talent_sd', 'chance_ra_se', 'total_r

# Pulling available game data

In [ ]:
print("Pulling availability data...")

AVAIL_YEARS = [2023, 2024, 2025]

# ── HITTER AVAILABILITY ───────────────────────────────────────
_hit_avail = pull_multi_year(mlb_batting, AVAIL_YEARS, 'G', id_col='playerid', qual=1)
g_cols = ['G_23', 'G_24', 'G_25']

_hit_avail['years_played'] = _hit_avail[g_cols].notna().sum(axis=1)
_hit_avail = _hit_avail[_hit_avail['years_played'] >= 2]
_hit_avail['availability'] = (_hit_avail[g_cols].mean(axis=1) / 162).clip(0.55, 1.0)
_hit_avail = _hit_avail[['playerid', 'availability']]
_hit_avail['playerid'] = _hit_avail['playerid'].astype('Int64')

hitters = hitters.drop(columns=['availability'], errors='ignore')
hitters['playerid'] = pd.to_numeric(hitters['playerid'], errors='coerce').astype('Int64')
hitters = hitters.merge(_hit_avail, on='playerid', how='left')
hitters['availability'] = hitters['availability'].fillna(0.85)

print(f"  Hitters — {hitters['availability'].ne(0.85).sum()} matched, "
      f"{hitters['availability'].lt(0.75).sum()} flagged as injury risks (<75%)")

# ── PITCHER AVAILABILITY ──────────────────────────────────────
_pit_avail = pull_multi_year(mlb_pitching, AVAIL_YEARS, 'IP', id_col='playerid', qual=1)
ip_cols = ['IP_23', 'IP_24', 'IP_25']

_pit_avail['years_played'] = _pit_avail[ip_cols].notna().sum(axis=1)
_pit_avail = _pit_avail[_pit_avail['years_played'] >= 2]
_pit_avail['avg_ip'] = _pit_avail[ip_cols].mean(axis=1)
_pit_avail['is_starter'] = _pit_avail['avg_ip'] > 50
_pit_avail['availability'] = np.where(
    _pit_avail['is_starter'],
    (_pit_avail['avg_ip'] / 170).clip(0.55, 1.0),
    (_pit_avail['avg_ip'] / 65).clip(0.55, 1.0)
)
_pit_avail = _pit_avail[['playerid', 'availability']]
_pit_avail['playerid'] = _pit_avail['playerid'].astype('Int64')

pitchers = pitchers.drop(columns=['availability'], errors='ignore')
pitchers['playerid'] = pd.to_numeric(pitchers['playerid'], errors='coerce').astype('Int64')
pitchers = pitchers.merge(_pit_avail, on='playerid', how='left')
pitchers['availability'] = pitchers['availability'].fillna(0.85)

Pulling availability data...
  Hitters — 0 matched, 0 flagged as injury risks (<75%)


In [ ]:
# Build MLBAM → FG ID bridge
id_bridge = pd.concat([
    hitters[['playerid', 'xMLBAMID']],
    pitchers[['playerid', 'xMLBAMID']]
]).drop_duplicates()
id_bridge['playerid'] = pd.to_numeric(id_bridge['playerid'], errors='coerce').astype('Int64')
id_bridge['xMLBAMID'] = pd.to_numeric(id_bridge['xMLBAMID'], errors='coerce').astype('Int64')

# Translate hitter IDs
_hit_avail = _hit_avail.rename(columns={'playerid': 'xMLBAMID'})
_hit_avail['xMLBAMID'] = _hit_avail['xMLBAMID'].astype('Int64')
_hit_avail = _hit_avail.merge(id_bridge, on='xMLBAMID', how='inner')

# Translate pitcher IDs
_pit_avail = _pit_avail.rename(columns={'playerid': 'xMLBAMID'})
_pit_avail['xMLBAMID'] = _pit_avail['xMLBAMID'].astype('Int64')
_pit_avail = _pit_avail.merge(id_bridge, on='xMLBAMID', how='inner')

# Scoring

In [ ]:
# ── HITTER SCORING (vectorized — fixes NaN bug) ──────────────────────────────
h = hitters.fillna(0)
hitters['fantasy_pts'] = (
    h['1B']  * 1.0   +
    h['2B']  * 2.0   +
    h['3B']  * 3.0   +
    h['HR']  * 4.0   +
    h['RBI'] * 1.0   +
    h['R']   * 1.0   +
    h['SB']  * 2.0   +
    h['BB']  * 1.0   +
    h['HBP'] * 0.5   +
    h['SO']  * -0.5  +
    h['CS']  * -0.5  +
    h['GDP'] * -0.5
)

# ── PITCHER SCORING (vectorized — QS and BS direct from data) ────────────────
p = pitchers.fillna(0)
pitchers['fantasy_pts'] = (
    p['IP']  * 1.25   +
    p['ER']  * -1.5   +
    p['H']   * -0.625 +
    p['BB']  * -0.625 +
    p['SO']  * 1.125  +
    p['W']   * 3.0    +
    p['L']   * -2.75  +
    p['SV']  * 4.0    +
    p['HLD'] * 2.5    +
    p['BS']  * -2.75  +
    p['QS']  * 5.0    +
    p['HBP'] * -0.5
)

# ── RANKED OUTPUTS ───────────────────────────────────────────────────────────
hit_ranked = hitters[['PlayerName', 'Team', 'Pos', 'PA', 'HR', 'SB', 'BB', 'SO', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
hit_ranked.index += 1

pit_ranked = pitchers[['PlayerName', 'Team', 'GS', 'G', 'IP', 'ERA', 'SO', 'SV', 'HLD', 'QS', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
pit_ranked.index += 1

print("=== TOP 30 HITTERS ===")
print(hit_ranked.head(30).to_string())

print("\n=== TOP 30 PITCHERS ===")
print(pit_ranked.head(30).to_string())

# ── SAVE ─────────────────────────────────────────────────────────────────────
hit_ranked.to_csv('hitter_rankings_2026.csv', index_label='Rank')
pit_ranked.to_csv('pitcher_rankings_2026.csv', index_label='Rank')

print("\n✅ CSVs saved: hitter_rankings_2026.csv & pitcher_rankings_2026.csv")


=== TOP 30 HITTERS ===
               PlayerName Team       Pos       PA       HR       SB        BB        SO  fantasy_pts
1           Shohei Ohtani  LAD  0.900862  658.176  43.5998  22.2882   91.4865  160.5680    594.62955
2             Aaron Judge  NYY  0.299517  633.319  42.2973   8.8028  112.2240  156.1930    561.12430
3               Juan Soto  NYM  0.111111  621.171  34.2543  19.6569  117.1530  110.5420    556.64725
4        Ronald Acuña Jr.  ATL  0.051095  676.219  30.8162  24.1564   90.7485  135.4620    552.57505
5          Bobby Witt Jr.  KCR  0.018692  633.655  27.9745  31.2121   47.0172  105.8900    540.06155
6   Vladimir Guerrero Jr.  TOR  0.142180  636.639  32.1766   4.7888   73.6591   87.5248    527.11065
7        Gunnar Henderson  BAL  0.057692  665.535  28.1358  23.5712   70.8129  134.2250    519.67665
8            José Ramírez  CLE  0.188073  603.880  26.8916  29.7858   58.2744   72.4692    514.40560
9          Corbin Carroll  ARI  0.030612  609.917  26.6738  32.5637 

# Master Board

In [ ]:
# Tag each group so we know the player type after merging
hitters['player_type']  = 'Hitter'
pitchers['player_type'] = 'Pitcher'

# Pitchers don't have a Pos column like hitters do — assign SP or RP
# based on whether they have 5+ projected starts
pitchers['Pos'] = pitchers.apply(
    lambda r: 'SP' if r['GS'] >= 5 else 'RP', axis=1
)

# ── FIX HITTER POS — map minpos to clean position label ──────────────────────
pos_map = {
    'C': 'C', '1B': '1B', '2B': '2B', '3B': '3B', 'SS': 'SS',
    'LF': 'OF', 'CF': 'OF', 'RF': 'OF', 'OF': 'OF',
    'DH': 'DH', 'UT': 'UTIL'
}

hitters['Pos'] = hitters['minpos'].str.split('/').str[0].map(pos_map).fillna('UTIL')

# Columns we want in the combined board
hitter_shared  = ['PlayerName', 'Team', 'Pos', 'player_type', 'fantasy_pts', 'ADP', 'xMLBAMID', 'availability', 'PA']
pitcher_shared = ['PlayerName', 'Team', 'Pos', 'player_type', 'fantasy_pts', 'ADP', 'xMLBAMID', 'availability', 'IP']

master = pd.concat([
    hitters[hitter_shared],
    pitchers[pitcher_shared]
]).sort_values('fantasy_pts', ascending=False).drop_duplicates(
    subset='xMLBAMID', keep='first'
).reset_index(drop=True)
master.index += 1

# Positive value = market is sleeping on this player relative to your scoring
master['value_vs_adp'] = master['ADP'] - master.index

undervalued = master[(master['value_vs_adp'] > 10) & (master['ADP'] < 700)]\
    .sort_values('value_vs_adp', ascending=False)


print("=== BIGGEST VALUES vs ADP ===")
print(undervalued.head(20).to_string())

print(master.head(30).to_string())
master.to_csv('master_draft_board_2026.csv', index_label='Overall_Rank')
print("\n✅ master_draft_board_2026.csv saved")


=== BIGGEST VALUES vs ADP ===
              PlayerName Team Pos player_type  fantasy_pts         ADP  xMLBAMID  availability       PA  IP  value_vs_adp
93         Nolan Arenado  ARI  3B      Hitter    335.50385  638.059998  571448.0          0.85  530.553 NaN    545.059998
71        Nolan Schanuel  LAA  1B      Hitter    373.67085  594.659973  694384.0          0.85  558.658 NaN    523.659973
119            Josh Bell  MIN  1B      Hitter    313.80690  636.330017  605137.0          0.85  486.908 NaN    517.330017
146       Chase Meidroth  CHW  2B      Hitter    298.68640  646.869995  805367.0          0.85  482.553 NaN    500.869995
203  Lourdes Gurriel Jr.  ARI  OF      Hitter    253.35560  689.070007  666971.0          0.85  386.133 NaN    486.070007
182         Mark Vientos  NYM  3B      Hitter    272.04755  652.570007  668901.0          0.85  432.092 NaN    470.570007
179      Jordan Westburg  BAL  2B      Hitter    275.58305  646.840027  676059.0          0.85  413.751 NaN    467.8

# Adding Savant Percentiles

In [ ]:
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks

# Clear prior columns (safe for re-runs)
master = master.drop(columns=['savant_mult', 'savant_score'], errors='ignore')

batter_savant_cols = ['xba', 'xslg', 'xwoba', 'xiso', 'xobp', 'brl', 'brl_percent',
                      'exit_velocity', 'max_ev', 'hard_hit_percent',
                      'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent']

pitcher_savant_cols = ['xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent',
                       'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent',
                       'bb_percent', 'whiff_percent', 'chase_percent', 'xera',
                       'fb_velocity', 'fb_spin', 'curve_spin']

# ── PULL 3 YEARS FOR BATTERS ──────────────────────────────────────────────────
_batter_raw = pull_savant_multi_year(
    statcast_batter_percentile_ranks, [2023, 2024, 2025], batter_savant_cols
)
print(f"✅ Batter Savant pulled — {len(_batter_raw)} unique batters across 3 years")

# ── PULL 3 YEARS FOR PITCHERS ─────────────────────────────────────────────────
_pitcher_raw = pull_savant_multi_year(
    statcast_pitcher_percentile_ranks, [2023, 2024, 2025], pitcher_savant_cols
)
print(f"✅ Pitcher Savant pulled — {len(_pitcher_raw)} unique pitchers across 3 years")

# ── HITTER SAVANT SCORE ───────────────────────────────────────────────────────
def pct_rank(series):
    return series.rank(pct=True) * 100

_batter_raw['savant_score'] = (
    pct_rank(_batter_raw['xba']) +
    pct_rank(_batter_raw['xslg']) +
    pct_rank(_batter_raw['brl_percent']) +
    pct_rank(_batter_raw['hard_hit_percent']) +
    pct_rank(_batter_raw['exit_velocity']) +
    pct_rank(_batter_raw['xwoba'])
).div(6)

_batter_raw['savant_mult'] = 1.0 + (_batter_raw['savant_score'] - 50) / 250

# CLIPPING + RECENCY (now flows to merge)
_batter_raw['savant_mult'] = np.clip(_batter_raw['savant_mult'], 0.75, 1.25)

batter_savant = _batter_raw[['player_id', 'savant_mult']].rename(columns={'player_id': 'xMLBAMID'})

# ── PITCHER SAVANT SCORE ──────────────────────────────────────────────────────
_pitcher_raw['savant_score'] = (
    pct_rank(-_pitcher_raw['xera']) +
    pct_rank(_pitcher_raw['k_percent']) +
    pct_rank(_pitcher_raw['whiff_percent']) +
    pct_rank(-_pitcher_raw['bb_percent']) +
    pct_rank(-_pitcher_raw['exit_velocity']) +
    pct_rank(-_pitcher_raw['hard_hit_percent'])
).div(6)

_pitcher_raw['savant_mult'] = 1.0 + (_pitcher_raw['savant_score'] - 50) / 250

# CLIPPING + RECENCY
_pitcher_raw['savant_mult'] = np.clip(_pitcher_raw['savant_mult'], 0.75, 1.25)
_pitcher_raw['savant_mult'] *= 1.05

pitcher_savant = _pitcher_raw[['player_id', 'savant_mult']].rename(columns={'player_id': 'xMLBAMID'})

# ── MERGE INTO MASTER ─────────────────────────────────────────────────────────
savant_all = pd.concat([batter_savant, pitcher_savant]).drop_duplicates('xMLBAMID')
master = master.merge(savant_all, on='xMLBAMID', how='left')
master['savant_mult'] = master['savant_mult'].fillna(1.0)  # Simplified fillna

# ── PERSONAL SAVANT OVERRIDES (for players missing from percentile ranks) ─────
personal_savant = {
    'agustin ramirez': 1.15,  # limited MLB PA history, set manually
}

for name, mult in personal_savant.items():
    mask = master['PlayerName'].apply(cleanname) == name
    master.loc[mask, 'savant_mult'] = mult

print(f"✅ Savant merged — {master['savant_mult'].gt(1).sum()} players above average")
print(master[['PlayerName', 'savant_mult']].head(15).to_string())


✅ Batter Savant pulled — 842 unique batters across 3 years
✅ Pitcher Savant pulled — 1062 unique pitchers across 3 years
✅ Savant merged — 676 players above average
               PlayerName  savant_mult
0           Shohei Ohtani     1.196354
1             Aaron Judge     1.198524
2               Juan Soto     1.193142
3        Ronald Acuña Jr.     1.187066
4          Bobby Witt Jr.     1.171615
5   Vladimir Guerrero Jr.     1.178646
6        Gunnar Henderson     1.143576
7            José Ramírez     1.068229
8          Corbin Carroll     1.086545
9      Fernando Tatis Jr.     1.170833
10        Freddie Freeman     1.117708
11        Elly De La Cruz     1.048872
12        Julio Rodríguez     1.143663
13            Kyle Tucker     1.120573
14       Francisco Lindor     1.117535


In [ ]:
print("Savant stats post-merge:")
print(master['savant_mult'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
print(f"95th percentile boost: {master['savant_mult'].quantile(0.95):.3f}x")
print(f"5th percentile drag: {master['savant_mult'].quantile(0.05):.3f}x")


Savant stats post-merge:
count    9343.000000
mean        1.003308
std         0.027171
min         0.811111
5%          1.000000
25%         1.000000
50%         1.000000
75%         1.000000
95%         1.043203
max         1.200894
Name: savant_mult, dtype: float64
95th percentile boost: 1.043x
5th percentile drag: 1.000x


## Hitters breakout model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("Pulling counting stats + Statcast for enhanced breakout model...")

# 1. TRAINING DATA (3 YEAR HISTORY)
# ─────────────────────────────────────────────────────────────

# Build training data directly from Steamer — wRC+ is already a column
merged = hitters[['playerid', 'PA', 'HR', 'SB', 'wRC+', 'xMLBAMID']].copy()
merged = merged.rename(columns={
    'playerid': 'IDfg',
    'PA': 'pa_avg',
    'HR': 'hr_avg',
    'SB': 'sb_avg',
    'wRC+': 'wrc_avg'
})
merged['IDfg'] = pd.to_numeric(merged['IDfg'], errors='coerce').astype('Int64')
merged['breakout'] = (merged['wrc_avg'] >= 115).astype(int)
merged = merged.dropna(subset=['pa_avg', 'hr_avg', 'sb_avg', 'wrc_avg'])

print(f"Training: {len(merged)} players, {merged['breakout'].mean():.1%} breakout rate")

# 2. STATCAST DATA
# ─────────────────────────────────────────────────────────────

statcast_cols = [
    'player_id',
    'xba',
    'xslg',
    'brl_percent',
    'hard_hit_percent',
    'whiff_percent',
    'exit_velocity',
    'chase_percent'
]

statcast_df = _batter_raw[statcast_cols].copy()

statcast_df['player_id'] = pd.to_numeric(
    statcast_df['player_id'],
    errors='coerce'
).astype('Int64')

# merge statcast
train_data = merged.merge(
    statcast_df,
    left_on='xMLBAMID',
    right_on='player_id',
    how='left'
)

print(
    "Statcast training coverage:",
    train_data['xba'].notna().sum(),
    "/",
    len(train_data)
)

# statcast feature aliases
train_data['xba_t'] = train_data['xba']
train_data['xslg_t'] = train_data['xslg']
train_data['brl_t'] = train_data['brl_percent']
train_data['hh_t'] = train_data['hard_hit_percent']
train_data['ev_t'] = train_data['exit_velocity']
train_data['chase_t'] = train_data['chase_percent']

# power/speed feature
train_data['power_speed'] = train_data['hr_avg'] * train_data['sb_avg']

# statcast underperformance feature
train_data['underperf'] = train_data['xslg_t'] - train_data['xba_t']

# 3. MODEL FEATURES
# ─────────────────────────────────────────────────────────────

features_b = [
    'pa_avg',
    'hr_avg',
    'sb_avg',
    'power_speed',
    'xba_t',
    'xslg_t',
    'brl_t',
    'hh_t',
    'whiff_percent',
    'ev_t',
    'chase_t',
    'underperf'
]

X = train_data[features_b].copy()
X = X.fillna(X.median())

y = train_data['breakout']

# 4. TRAIN MODEL
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

clf_breakout = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

clf_breakout.fit(X_train,y_train)

print(f"✅ Enhanced model accuracy: {clf_breakout.score(X_test,y_test):.2%}")

print("\n📊 Feature Importance:")

importance_df = pd.DataFrame({
    'feature':features_b,
    'importance':clf_breakout.feature_importances_
}).sort_values('importance',ascending=False)

print(importance_df.round(3))

# 5. SCORE ALL HITTERS
# ─────────────────────────────────────────────────────────────

scoring_data = hitters[['playerid','PA','HR','SB','xMLBAMID']].rename(columns={
    'playerid':'IDfg', 'PA':'pa_avg', 'HR':'hr_avg', 'SB':'sb_avg'
})

scoring_data['IDfg'] = pd.to_numeric(scoring_data['IDfg'], errors='coerce').astype('Int64')
scoring_data['xMLBAMID'] = pd.to_numeric(scoring_data['xMLBAMID'], errors='coerce').astype('Int64')

# merge statcast
scoring_data = scoring_data.merge(
    statcast_df,
    left_on='xMLBAMID',
    right_on='player_id',
    how='left'
)

# statcast features
scoring_data['xba_t'] = scoring_data['xba']
scoring_data['xslg_t'] = scoring_data['xslg']
scoring_data['brl_t'] = scoring_data['brl_percent']
scoring_data['hh_t'] = scoring_data['hard_hit_percent']
scoring_data['ev_t'] = scoring_data['exit_velocity']
scoring_data['chase_t'] = scoring_data['chase_percent']

scoring_data['power_speed'] = scoring_data['hr_avg'] * scoring_data['sb_avg']

scoring_data['underperf'] = scoring_data['xslg_t'] - scoring_data['xba_t']

scoring_data[features_b] = scoring_data[features_b].fillna(X.median())

print(
    f"✅ Statcast coverage: "
    f"{scoring_data['xba'].notna().sum()}/{len(scoring_data)} "
    f"({scoring_data['xba'].notna().mean():.1%})"
)

# predict breakout probability
scoring_data['breakout_prob'] = clf_breakout.predict_proba(
    scoring_data[features_b]
)[:,1]

# 6. MERGE BACK INTO PROJECTIONS
# ─────────────────────────────────────────────────────────────

hitters['xMLBAMID'] = pd.to_numeric(hitters['xMLBAMID'], errors='coerce').astype('Int64')

# Drop existing breakout_prob to avoid merge collision
hitters = hitters.drop(columns=['breakout_prob'], errors='ignore')

hitters = hitters.merge(
    scoring_data[['xMLBAMID','breakout_prob']],
    on='xMLBAMID',
    how='left'
)

hitters['breakout_prob'] = hitters['breakout_prob'].fillna(0)

print(f"✅ {(hitters['breakout_prob']>0).sum()}/{len(hitters)} hitters scored")

# breakout score weighted by playing time
hitters['breakout_score'] = hitters['breakout_prob'] * np.log1p(hitters['PA'])

print("\nTop Breakout Candidates (Filtered):")

# Remove elite players
hitters['elite_filter'] = hitters['PA'] > 600

# Remove low playing time
hitters['pt_filter'] = hitters['PA'] < 200

candidate_pool = hitters.loc[
    (~hitters['elite_filter']) &
    (~hitters['pt_filter'])
].copy()

top_breakouts = (
    candidate_pool
    .nlargest(10,'breakout_score')
    [['PlayerName','Team','breakout_prob','breakout_score','PA']]
)

print(top_breakouts.round(3).to_string(index=False))

Pulling counting stats + Statcast for enhanced breakout model...
Training: 4186 players, 1.9% breakout rate
Statcast training coverage: 381 / 4186
✅ Enhanced model accuracy: 97.97%

📊 Feature Importance:
          feature  importance
0          pa_avg       0.246
1          hr_avg       0.242
3     power_speed       0.109
2          sb_avg       0.104
5          xslg_t       0.098
9            ev_t       0.075
4           xba_t       0.045
7            hh_t       0.039
6           brl_t       0.014
8   whiff_percent       0.013
11      underperf       0.009
10        chase_t       0.007
✅ Statcast coverage: 381/4186 (9.1%)
✅ 291/4186 hitters scored

Top Breakout Candidates (Filtered):
     PlayerName Team  breakout_prob  breakout_score      PA
  Manny Machado  SDP          0.996           6.358 591.132
    Ketel Marte  ARI          0.986           6.300 596.305
    Kyle Tucker  LAD          0.975           6.175 562.259
   Riley Greene  DET          0.964           6.141 582.170
   Bry

## Pitchers breakout model

In [ ]:
print("Pulling counting stats + Statcast for enhanced pitcher breakout model...")

# 1. TRAINING DATA (3 YEAR HISTORY)
# ─────────────────────────────────────────────────────────────

merged_p = pitchers[['playerid', 'IP', 'W', 'SO', 'FIP', 'xMLBAMID']].copy()
merged_p = merged_p.rename(columns={
    'playerid': 'IDfg',
    'IP': 'ip_avg',
    'W':  'w_avg',
    'SO': 'so_avg',
    'FIP': 'fip_avg'
})
merged_p['IDfg'] = pd.to_numeric(merged_p['IDfg'], errors='coerce').astype('Int64')
merged_p['breakout'] = (merged_p['fip_avg'] <= 3.75).astype(int)
merged_p = merged_p.dropna(subset=['ip_avg', 'so_avg', 'w_avg', 'fip_avg'])
merged_p = merged_p[merged_p['ip_avg'] >= 40]
print(f"Training: {len(merged_p)} pitchers, {merged_p['breakout'].mean():.1%} breakout rate")


# 2. STATCAST DATA
# ─────────────────────────────────────────────────────────────

statcast_cols_p = [
    'player_id',
    'xera',
    'xba',
    'xslg',
    'brl_percent',
    'hard_hit_percent',
    'exit_velocity',
    'k_percent',
    'bb_percent',
    'whiff_percent',
    'chase_percent',
    'fb_velocity',
    'fb_spin'
]

statcast_df_p = _pitcher_raw[statcast_cols_p].copy()

statcast_df_p['player_id'] = pd.to_numeric(
    statcast_df_p['player_id'],
    errors='coerce'
).astype('Int64')

# merge statcast
train_data_p = merged_p.merge(
    statcast_df_p,
    left_on='xMLBAMID',
    right_on='player_id',
    how='left'
)

print(
    "Statcast training coverage:",
    train_data_p['xera'].notna().sum(),
    "/",
    len(train_data_p)
)

# statcast feature aliases
train_data_p['xera_t']    = train_data_p['xera']
train_data_p['xba_t']     = train_data_p['xba']
train_data_p['xslg_t']    = train_data_p['xslg']
train_data_p['brl_t']     = train_data_p['brl_percent']
train_data_p['hh_t']      = train_data_p['hard_hit_percent']
train_data_p['ev_t']      = train_data_p['exit_velocity']
train_data_p['k_t']       = train_data_p['k_percent']
train_data_p['bb_t']      = train_data_p['bb_percent']
train_data_p['whiff_t']   = train_data_p['whiff_percent']
train_data_p['chase_t']   = train_data_p['chase_percent']
train_data_p['fbv_t']     = train_data_p['fb_velocity']
train_data_p['spin_t']    = train_data_p['fb_spin']

# stuff quality feature
train_data_p['stuff_score'] = train_data_p['k_t'] * train_data_p['fbv_t']

# contact suppression feature
train_data_p['contact_supp'] = train_data_p['xba_t'] + train_data_p['brl_t']

# 3. MODEL FEATURES
# ─────────────────────────────────────────────────────────────

features_p = [
    'ip_avg',
    'so_avg',
    'w_avg',
    'stuff_score',
    'xera_t',
    'xba_t',
    'xslg_t',
    'brl_t',
    'hh_t',
    'ev_t',
    'k_t',
    'bb_t',
    'whiff_t',
    'chase_t',
    'fbv_t',
    'spin_t',
    'contact_supp'
]

X_p = train_data_p[features_p].copy()
X_p = X_p.fillna(X_p.median())

y_p = train_data_p['breakout']

# 4. TRAIN MODEL
# ─────────────────────────────────────────────────────────────

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_p,
    y_p,
    test_size=0.2,
    random_state=42
)

clf_breakout_pit = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

clf_breakout_pit.fit(X_train_p, y_train_p)

print(f"Enhanced pitcher model accuracy: {clf_breakout_pit.score(X_test_p, y_test_p):.2%}")

print("\n Feature Importance:")

importance_df_p = pd.DataFrame({
    'feature':    features_p,
    'importance': clf_breakout_pit.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df_p.round(3))

# 5. SCORE ALL PITCHERS
# ─────────────────────────────────────────────────────────────

scoring_data_p = pitchers[['playerid','IP','W','SO','FIP','xMLBAMID']].rename(columns={
    'playerid':'IDfg', 'IP':'ip_avg', 'W':'w_avg', 'SO':'so_avg', 'FIP':'fip_avg'
})

scoring_data_p['IDfg']     = pd.to_numeric(scoring_data_p['IDfg'],     errors='coerce').astype('Int64')
scoring_data_p['xMLBAMID'] = pd.to_numeric(scoring_data_p['xMLBAMID'], errors='coerce').astype('Int64')

# Only score pitchers with meaningful projections
scoring_data_p = scoring_data_p[scoring_data_p['ip_avg'] >= 50].copy()

# merge statcast
scoring_data_p = scoring_data_p.merge(
    statcast_df_p,
    left_on='xMLBAMID',
    right_on='player_id',
    how='left'
)

# statcast features
scoring_data_p['xera_t']  = scoring_data_p['xera']
scoring_data_p['xba_t']   = scoring_data_p['xba']
scoring_data_p['xslg_t']  = scoring_data_p['xslg']
scoring_data_p['brl_t']   = scoring_data_p['brl_percent']
scoring_data_p['hh_t']    = scoring_data_p['hard_hit_percent']
scoring_data_p['ev_t']    = scoring_data_p['exit_velocity']
scoring_data_p['k_t']     = scoring_data_p['k_percent']
scoring_data_p['bb_t']    = scoring_data_p['bb_percent']
scoring_data_p['whiff_t'] = scoring_data_p['whiff_percent']
scoring_data_p['chase_t'] = scoring_data_p['chase_percent']
scoring_data_p['fbv_t']   = scoring_data_p['fb_velocity']
scoring_data_p['spin_t']  = scoring_data_p['fb_spin']

scoring_data_p['stuff_score']  = scoring_data_p['k_t']   * scoring_data_p['fbv_t']
scoring_data_p['contact_supp'] = scoring_data_p['xba_t'] + scoring_data_p['brl_t']

# Only keep pitchers with real Statcast data — no median filling for pitchers
valid_p = scoring_data_p[features_p].notna().all(axis=1)
scoring_data_p = scoring_data_p[valid_p].copy()

print(
    f"✅ Statcast coverage: "
    f"{scoring_data_p['xera'].notna().sum()}/{len(scoring_data_p)} "
    f"({scoring_data_p['xera'].notna().mean():.1%})"
)

# predict breakout probability
scoring_data_p['breakout_prob'] = clf_breakout_pit.predict_proba(
    scoring_data_p[features_p]
)[:, 1]

# 6. MERGE BACK INTO PROJECTIONS
# ─────────────────────────────────────────────────────────────

pitchers['xMLBAMID'] = pd.to_numeric(pitchers['xMLBAMID'], errors='coerce').astype('Int64')

# Drop existing breakout_prob to avoid merge collision
pitchers = pitchers.drop(columns=['breakout_prob'], errors='ignore')

pitchers = pitchers.merge(
    scoring_data_p[['xMLBAMID', 'breakout_prob']],
    on='xMLBAMID',
    how='left'
)

pitchers['breakout_prob'] = pitchers['breakout_prob'].fillna(0)

print(f"✅ {(pitchers['breakout_prob'] > 0).sum()}/{len(pitchers)} pitchers scored")

# breakout score weighted by innings
pitchers['breakout_score'] = pitchers['breakout_prob'] * pitchers['IP'].fillna(0)

print("\nTop Breakout Candidates (Filtered):")

candidate_pool_p = pitchers.loc[
    (pitchers['IP'] >= 40) &
    (pitchers['IP'] <= 160)
].copy()

top_breakouts_p = (
    candidate_pool_p
    .nlargest(10, 'breakout_score')
    [['PlayerName', 'Team', 'breakout_prob', 'breakout_score', 'IP']]
)

print(top_breakouts_p.round(3).to_string(index=False))

# ── MERGE BREAKOUT PROBS INTO MASTER ─────────────────────────────────────────
master = master.drop(columns=['breakout_prob'], errors='ignore')
master['xMLBAMID'] = pd.to_numeric(master['xMLBAMID'], errors='coerce').astype('Int64')

# Build combined breakout map from both hitters and pitchers
hitter_bp  = hitters[['xMLBAMID', 'breakout_prob']].drop_duplicates('xMLBAMID').copy()
pitcher_bp = pitchers[['xMLBAMID', 'breakout_prob']].drop_duplicates('xMLBAMID').copy()
all_bp = pd.concat([hitter_bp, pitcher_bp]).drop_duplicates('xMLBAMID')

master = master.merge(all_bp, on='xMLBAMID', how='left')
master['breakout_prob'] = master['breakout_prob'].fillna(0.0)

print(f"✅ breakout_prob merged into master — {master['breakout_prob'].gt(0).sum()} players scored")
print(master[['PlayerName', 'player_type', 'breakout_prob']].head(20).to_string())

Pulling counting stats + Statcast for enhanced pitcher breakout model...
Training: 514 pitchers, 18.5% breakout rate
Statcast training coverage: 377 / 514
Enhanced pitcher model accuracy: 94.17%

 Feature Importance:
         feature  importance
3    stuff_score       0.123
14         fbv_t       0.096
2          w_avg       0.096
1         so_avg       0.094
10           k_t       0.089
4         xera_t       0.074
12       whiff_t       0.061
0         ip_avg       0.057
13       chase_t       0.057
6         xslg_t       0.046
15        spin_t       0.040
11          bb_t       0.035
16  contact_supp       0.031
9           ev_t       0.027
5          xba_t       0.026
8           hh_t       0.024
7          brl_t       0.024
✅ Statcast coverage: 352/352 (100.0%)
✅ 345/5162 pitchers scored

Top Breakout Candidates (Filtered):
        PlayerName Team  breakout_prob  breakout_score      IP
Yoshinobu Yamamoto  LAD          0.899         138.813 154.428
   Spencer Strider  ATL          

In [ ]:
# ── DIAGNOSTIC: Breakout Prob Sanity Check ────────────────────────────────────

# 1. Check spread — should NOT be a flat value
print("=== BREAKOUT_PROB DISTRIBUTION ===")
print(master['breakout_prob'].describe())
print(f"\nUnique values: {master['breakout_prob'].nunique()}")
print(f"Players stuck at 0.531: {(master['breakout_prob'] == 0.531).sum()}")
print(f"Players stuck at 0.000: {(master['breakout_prob'] == 0.000).sum()}")

# 2. Spot-check young breakout candidates vs veterans
print("\n=== SPOT CHECK: TARGET PLAYERS ===")
targets = ['Jackson Chourio', 'Julio Rodríguez', 'Ceddanne Rafaela',
           'Alex Bregman', 'Tarik Skubal', 'Yordan Alvarez', 'Max Fried']
print(master[master['PlayerName'].isin(targets)]
      [['PlayerName', 'player_type', 'breakout_prob', 'savant_mult']]
      .sort_values('breakout_prob', ascending=False)
      .to_string())

# 3. Top 15 highest breakout prob hitters (should be young/high-contact-quality)
print("\n=== TOP 15 HITTER BREAKOUT CANDIDATES ===")
print(master[(master['player_type'] == 'Hitter') & (master['breakout_prob'] > 0)]
      .nlargest(15, 'breakout_prob')
      [['PlayerName', 'breakout_prob', 'savant_mult']]
      .to_string())

# 4. Top 15 highest breakout prob pitchers
print("\n=== TOP 15 PITCHER BREAKOUT CANDIDATES ===")
print(master[(master['player_type'] == 'Pitcher') & (master['breakout_prob'] > 0)]
      .nlargest(15, 'breakout_prob')
      [['PlayerName', 'breakout_prob', 'savant_mult']]
      .to_string())

# 5. Merge miss check — how many got filled with 0.0?
hitter_zeros  = master[(master['player_type'] == 'Hitter')  & (master['breakout_prob'] == 0.0)]
pitcher_zeros = master[(master['player_type'] == 'Pitcher') & (master['breakout_prob'] == 0.0)]
print(f"\n=== MERGE MISS CHECK ===")
print(f"Hitters  with breakout_prob = 0.0: {len(hitter_zeros)}")
print(f"Pitchers with breakout_prob = 0.0: {len(pitcher_zeros)}")
if len(hitter_zeros) > 0:
    print("Sample hitter misses:", hitter_zeros['PlayerName'].head(10).tolist())
if len(pitcher_zeros) > 0:
    print("Sample pitcher misses:", pitcher_zeros['PlayerName'].head(10).tolist())


=== BREAKOUT_PROB DISTRIBUTION ===
count    9343.000000
mean        0.018413
std         0.111258
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.996108
Name: breakout_prob, dtype: float64

Unique values: 555
Players stuck at 0.531: 0
Players stuck at 0.000: 8708

=== SPOT CHECK: TARGET PLAYERS ===
           PlayerName player_type  breakout_prob  savant_mult
17     Yordan Alvarez      Hitter       0.939133     1.192014
30       Tarik Skubal     Pitcher       0.902276     1.000513
25       Alex Bregman      Hitter       0.843352     1.035069
143         Max Fried     Pitcher       0.835090     0.960993
12    Julio Rodríguez      Hitter       0.742454     1.143663
28    Jackson Chourio      Hitter       0.740350     1.040972
101  Ceddanne Rafaela      Hitter       0.006363     0.925521

=== TOP 15 HITTER BREAKOUT CANDIDATES ===
               PlayerName  breakout_prob  savant_mult
37          Manny Machado       0.995976     1.142708
48 

# Importing Prospect and Dynasty Rankings

In [ ]:
import gspread
from google.colab import auth
from google.auth import default

# This is the correct auth method for Colab — triggers a one-time popup
auth.authenticate_user()

creds, _ = default(scopes=[
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
])

gc = gspread.authorize(creds)
print("✅ Google Sheets authenticated")


✅ Google Sheets authenticated


In [ ]:
# ── PROSPECT DATA — load from full CSV instead of Google Sheet ────────────────
prospect_data = pd.read_csv('/content/drive/MyDrive/2026 Top Prospects Fangraphs - Sheet1.csv')

# Clean FV — strips "40+" → 40, "45+" → 45, etc.
prospect_data['FV'] = prospect_data['FV'].astype(str).str.replace('+', '', regex=False)
prospect_data['FV'] = pd.to_numeric(prospect_data['FV'], errors='coerce').fillna(40).astype(int)

# Add 'Nameclean' column to prospect_data
prospect_data['Name_clean'] = prospect_data['Name'].str.strip().str.lower()

# ── FANTRAX HQ RANKINGS — load from local CSV──────
top500_hq = pd.read_csv('/content/drive/MyDrive/Top-500 Fantrax Dynasty Rankings February 2026.csv')  # file:34
top500_hq = top500_hq.rename(columns={'Points': 'HQ_Rank', 'Pos.': 'Pos', 'Player': 'PlayerName'})
top500_hq['Name_clean'] = top500_hq['PlayerName'].str.strip().str.lower()

print("✅ Prospect sheet loaded:", len(prospect_data), "players")
print("✅ HQ Top 500 loaded:", len(top500_hq), "players")
print("HQ #1:", top500_hq.iloc[0]['PlayerName'])  # Shohei Ohtani

✅ Prospect sheet loaded: 200 players
✅ HQ Top 500 loaded: 503 players
HQ #1: Shohei Ohtani


In [ ]:
# ── PROSPECT DATA ─────────────────────────────────────────────────────────────
prospect_data['Name_clean'] = prospect_data['Name'].apply(cleanname)

# ── TOP 500 ───────────────────────────────────────────────────────────────────
top500_hq['Name_clean'] = top500_hq['PlayerName'].apply(cleanname)
# ── BA DYNASTY TOP 500 ────────────────────────────────────────────────────────
ba_dynasty = pd.read_csv('/content/drive/MyDrive/BA Dynasty Top 500.csv')
ba_dynasty = ba_dynasty.rename(columns={'PLAYER': 'PlayerName', 'RANK': 'BARank'})
ba_dynasty['Nameclean'] = ba_dynasty['PlayerName'].apply(cleanname)
print(f"BA Dynasty Top 500 loaded — {len(ba_dynasty)} players")

BA Dynasty Top 500 loaded — 500 players


# League Teams

In [ ]:
# Load the Fantrax league export you uploaded to Colab
fantrax = pd.read_csv('/content/drive/MyDrive/Fantrax Players 2026 Draft.csv')

# Convert numeric columns — errors='coerce' turns any bad values into NaN
fantrax['FPts'] = pd.to_numeric(fantrax['FPts'], errors='coerce').fillna(0)
fantrax['Age']  = pd.to_numeric(fantrax['Age'],  errors='coerce')
fantrax['ADP']  = pd.to_numeric(fantrax['ADP'],  errors='coerce')

# Map Fantrax's short team abbreviations to full team names
team_map = {
    'SEAMHEAD' : 'The Seamhead Express',
    'HoF'      : 'Hoos on First',
    'WOOD'     : 'James Wood Truthers',
    'SHW'      : 'The Boston Red Shawx',
    'MBC'      : "Madoff's Biggest Client",
    'ZL'       : 'New Team 6',
    'AT'       : 'Orioles Catchers Club',
    'MB'       : 'Markyball',
    'Luka'     : 'Mick Redemption Rally',
    'DEAL'     : 'Wine and Dynasty',
    'SW'      : "Spinmaster Wizards",
    'HCGA'     : "Harry Caray's Green Apples",
    'Nunez'    : 'Baltimore KnOrioles',
    '5B DSC'   : '$5B Dollar Shave Club',

}

fantrax['FantasyTeam'] = fantrax['Status'].map(team_map).fillna('FA')

# Keeper Template

In [ ]:
'''

# Generate a blank keeper template for all 14 teams
teams = [
    "Harry Caray's Green Apples", "Markyball", "morning WOOD",
    "Wine and Dynasty", "Madoff's Biggest Client", "Spinmaster Wizards",
    "The Boston Red Shawx", "Hoos on First", "The Seamhead Express",
    "Ashcrafts or Tatis", "Mick Redemption Rally",
    "New Team 6", "$5B Dollar Shave Club", "Baltimore KnOrioles"
]

# Create empty template
keeper_template = pd.DataFrame({
    'PlayerName': [''] * 22 * 14,
    'FantasyTeam': [t for t in teams for _ in range(22)],
    'IsProspect':  [False] * 22 * 14   # flag minor leaguers
})

keeper_template.to_csv('keeper_template.csv', index=False)
print("✅ keeper_template.csv saved — fill this in and re-upload")

'''


'\n\n# Generate a blank keeper template for all 14 teams\nteams = [\n    "Harry Caray\'s Green Apples", "Markyball", "morning WOOD",\n    "Wine and Dynasty", "Madoff\'s Biggest Client", "Spinmaster Wizards",\n    "The Boston Red Shawx", "Hoos on First", "The Seamhead Express",\n    "Ashcrafts or Tatis", "Mick Redemption Rally",\n    "New Team 6", "$5B Dollar Shave Club", "Baltimore KnOrioles"\n]\n\n# Create empty template\nkeeper_template = pd.DataFrame({\n    \'PlayerName\': [\'\'] * 22 * 14,\n    \'FantasyTeam\': [t for t in teams for _ in range(22)],\n    \'IsProspect\':  [False] * 22 * 14   # flag minor leaguers\n})\n\nkeeper_template.to_csv(\'keeper_template.csv\', index=False)\nprint("✅ keeper_template.csv saved — fill this in and re-upload")\n\n'

# Value Scores

In [ ]:
## Value scores
# ── FV GRADE → BASE SCORE ─────────────────────────────────────────────────────
fv_to_score = {80: 500, 70: 400, 65: 350, 60: 300, 55: 240, 50: 180, 45: 120, 40: 80}
# ── YOUR PERSONAL FV BOOSTS ─────────────────────────────────────────────────── ← ADD HERE
personal_fv = {
}
# ── PROSPECT FV LOOKUP (was missing from notebook) ────────────────────────────
def get_prospect_fv(player_name, default_fv=40):
    key = cleanname(player_name)
    if key in personal_fv:
        return personal_fv[key]
    match = prospect_data[prospect_data['Name_clean'] == cleanname(player_name)]
    if not match.empty:
        return int(match.iloc[0]['FV'])
    return default_fv


# ── DYNASTY RANK LOOKUP (FantraxHQ Top 500) ────────────────────────────────────
top500_data = top500_hq
def get_player_rank(player_name, default_rank=999):
    clean_player = cleanname(player_name)

    # Check Baseball America Rankings
    ba_match = ba_dynasty[ba_dynasty['Nameclean'] == clean_player]
    if not ba_match.empty:
        return int(ba_match.iloc[0]['BARank'])

    # Check FantraxHQ first (most current)
    hq_match = top500_hq[top500_hq['Name_clean'] == clean_player]
    if not hq_match.empty:
        return int(hq_match.iloc[0]['HQ_Rank'])

    # Fallback to your existing top500_data
    match = top500_data[top500_data['Name_clean'] == clean_player]
    if not match.empty:
        return int(match.iloc[0]['Points'])

    return default_rank


# ── RANK BONUS — rewards rank 1 vs rank 100 within same FV tier ──────────────
def rank_bonus(rank, max_rank=110):
    if rank >= 999:   return 0
    if rank <=25:     return 50
    if rank <=100:    return 25
    if rank <= 200:   return 15
    return max(0, round(1 - (rank - 200) / 300, 0))

# ── Players to evaluate using FV instead of FPts (partial MLB stats skewing) ──
# Add any player here whose FPts underrepresent their true dynasty value
fv_override = {'Max Clark', 'Chase Dollander', 'Logan Henderson',
               'Owen Caissie', 'Sal Stewart', 'Jesús Made'}
injury_override = {'Ronald Acuña Jr.'}

# ── AGE MULTIPLIER (Added to fix NameError) ───────────────────────────────────
def dynasty_age_multiplier(age):
    if pd.isna(age):  # Handle NaN ages, e.g., for new prospects without age data
        return 1.0
    if age <= 23: return 1.10
    if age <= 26: return 1.05
    if age <= 30: return 1.00
    if age <= 33: return 0.95
    if age <= 35: return 0.90
    return 0.85

# ── UNIFIED VALUE SCORE ───────────────────────────────────────────────────────
def calc_value_score(row):
    # ── SAFE GETTERS ──────────────────────────────────────────────────────────
    age         = row.get('Age', 28);           age         = 28.0 if pd.isna(age)         else float(age)
    fpts        = row.get('FPts', 0);           fpts        = 0.0  if pd.isna(fpts)        else float(fpts)
    fantasy_pts = row.get('fantasy_pts', 0);    fantasy_pts = 0.0  if pd.isna(fantasy_pts) else float(fantasy_pts)
    player_name = row.get('PlayerName', '')
    savant      = row.get('savant_mult', 1.0);  savant      = 1.0  if pd.isna(savant)      else float(savant)
    breakout    = row.get('breakout_prob', 0.0); breakout   = 0.0  if pd.isna(breakout)    else float(breakout)

    age_mult       = dynasty_age_multiplier(age)
    pure_prospect  = fpts < 100
    has_real_stats = fpts >= 300
    low_projection = fantasy_pts < 50

    # ── CHECK YOUR PROSPECT SHEET ─────────────────────────────────────────────
    prospect_match = prospect_data[prospect_data['Name_clean'] == cleanname(player_name)]
    has_fv_data    = not prospect_match.empty
    prospect_fv    = int(prospect_match.iloc[0]['FV']) if has_fv_data else 40

    # ── FV PATH TRIGGER ───────────────────────────────────────────────────────
    use_fv = (
        (pure_prospect and low_projection)
        or (has_fv_data and prospect_fv >= 50 and not has_real_stats)
        or (cleanname(player_name) in {cleanname(p) for p in fv_override} and not has_real_stats)
    ) and cleanname(player_name) not in {cleanname(p) for p in injury_override}

    if use_fv:
        fv   = get_prospect_fv(player_name)   # checks personal_fv first
        nearest_fv = max((k for k in fv_to_score if k <= fv), default=40)
        base = fv_to_score.get(fv, fv_to_score[nearest_fv])
        # HQ rank first, then prospect sheet Top 100, then 999
        hq_match = top500_hq[top500_hq['Name_clean'] == cleanname(player_name)] # Corrected variable and column name
        if not hq_match.empty:
            rk = int(hq_match.iloc[0]['HQ_Rank'])
        elif has_fv_data and pd.notna(prospect_match.iloc[0].get('Top 100', None)):
            rk = int(prospect_match.iloc[0]['Top 100'])
        else:
            rk = 999
        bonus = rank_bonus(rk)
        return (base + bonus) * age_mult

    else:
        blended = (0.7 * fantasy_pts) + (0.3 * fpts)

        pos = row.get('Pos', '') or row.get('Position', '')
        if pos == 'SP':
            blended *= 1.25
            if fantasy_pts >= 380:
                blended += 50
        elif pos == 'RP':
            blended *= 1.10
        # ── OPTION 2: availability discount ──────────────────────────
        availability = float(row.get('availability', 0.88))
        blended *= availability

        # ── OPTION 3: PA/IP floor penalty ────────────────────────────
        proj_pa = float(row.get('PA', 500))
        proj_ip = float(row.get('IP', 170))

        if pos == 'SP' and proj_ip < 130:
            blended *= (proj_ip / 150)   # e.g. 90 IP → 0.60x penalty
        elif pos not in ('SP', 'RP') and proj_pa < 400:
            blended *= (proj_pa / 450)   # e.g. 300 PA → 0.67x penalty

        if blended > 400 and age > 30:
            age_mult = max(age_mult, 0.95)

        savant        = 1.0 + (savant - 1.0) * 0.6
        breakout_mult = 1.0 + (breakout * 0.02)

        star_floor = 0
        if blended >= 550: star_floor = 20
        elif blended >= 500: star_floor = 10

        return (blended * age_mult * savant * breakout_mult) + star_floor

# Initialize my_roster using a filtered fantrax DataFrame (e.g., for 'The Seamhead Express')
# In a real scenario, you would load your keeper_template.csv here after filling it out.
my_roster = fantrax[fantrax['FantasyTeam'] == 'The Seamhead Express'].copy()
my_roster = my_roster.rename(columns={'Player': 'PlayerName'})


# ── MERGE PROJECTED STATS INTO ROSTER ────────────────────────────────────────
if 'fantasy_pts' not in my_roster.columns:
    my_roster = my_roster.merge(
        master[['PlayerName', 'fantasy_pts']],
        on='PlayerName', how='left'
    )
# ── MERGE POS FROM MASTER INTO ROSTER ────────────────────────────────────────
pos_map = master[['PlayerName', 'Pos']].drop_duplicates('PlayerName')
my_roster = my_roster.merge(pos_map, on='PlayerName', how='left')
my_roster['Pos'] = my_roster['Pos'].fillna('')

# ── MERGE SAVANT MULTIPLIER INTO ROSTER ──────────────────────────────────────
if 'savant_mult' not in my_roster.columns:
    my_roster = my_roster.merge(
        master[['PlayerName', 'savant_mult']],
        on='PlayerName', how='left'
    )
    my_roster['savant_mult'] = my_roster['savant_mult'].fillna(1.0)

# ── MERGE PA, IP, AVAILABILITY, BREAKOUT INTO ROSTER ─────────────────────────
roster_extras = master[['PlayerName', 'PA', 'IP', 'availability', 'breakout_prob']].drop_duplicates('PlayerName')
my_roster = my_roster.merge(roster_extras, on='PlayerName', how='left')
my_roster['availability']  = my_roster['availability'].fillna(0.85)
my_roster['breakout_prob'] = my_roster['breakout_prob'].fillna(0.0)
my_roster['PA']            = my_roster['PA'].fillna(500)
my_roster['IP']            = my_roster['IP'].fillna(170)

# ── CALCULATE VALUE SCORES ────────────────────────────────────────────────────
my_roster['value_score'] = my_roster.apply(calc_value_score, axis=1)
my_roster = my_roster.sort_values('value_score', ascending=False).reset_index(drop=True)
my_roster.index += 1
my_roster['KEEP'] = my_roster.index <= 22


print(my_roster[['PlayerName', 'Position', 'Age', 'FPts', 'fantasy_pts', 'value_score', 'KEEP']].to_string())
my_roster.to_csv('/content/drive/MyDrive/WeLoveSabermetrics/seamhead_keeper_decisions.csv', index_label='Rank')
print("\n✅ Saved to Drive")

         PlayerName Position  Age    FPts  fantasy_pts  value_score  KEEP
1       Aaron Judge       OF   33  677.00   561.124300   557.675579  True
2   Freddie Freeman       1B   36  453.00   478.639700   415.003411  True
3         Max Clark       OF   21   47.50     1.289350   357.500000  True
4      Brice Turang       2B   26  445.50   398.295500   354.685666  True
5          Ian Happ       OF   31  436.50   397.765700   348.819490  True
6          Ben Rice     C,1B   27  394.50   331.412000   337.959665  True
7    Bryan Reynolds       OF   31  402.50   365.731500   331.518079  True
8      Matt Chapman       3B   32  381.50   381.594250   330.682198  True
9   Shea Langeliers        C   28  381.50   348.183350   320.769762  True
10  Brendan Donovan       2B   29  340.00   343.136600   305.407345  True
11   Freddy Peralta       SP   29  309.25   262.394825   304.763917  True
12  Xander Bogaerts       SS   33  382.00   334.076900   275.813175  True
13      Sal Stewart       1B   22  305

In [ ]:
# ── LEAGUE-WIDE VALUE SCORES─────────────────────────────

# 1. Start from clean master (already has Pos, fantasy_pts, ADP, savant_mult, etc.)
league = master.copy()

# 2. Add missing cols with defaults (no merge needed)
league['FPts'] = 0.0
league['Age'] = league['Age'] if 'Age' in league.columns else 28
league['FantasyTeam'] = 'FA'
league['value_score'] = league['fantasy_pts'] # Base

# 3. Apply multipliers (if exist)
if 'savant_mult' in league.columns:
    league['value_score'] *= league['savant_mult'].fillna(1.0)
if 'breakout_prob' in league.columns:
    league['value_score'] *= (1 + league['breakout_prob'] * 0.05)

# 4. Safe Fantrax merge (optional, won't break)
try:
    fan_merge = fantrax[['Player', 'FPts', 'Age', 'FantasyTeam']].copy()
    fan_merge.columns = ['PlayerName', 'FPts', 'Age', 'FantasyTeam']
    league = league.merge(fan_merge, on='PlayerName', how='left', suffixes=('', '_f'))
    league['FPts'] = league['FPts_f'].fillna(league['FPts'])
    league['Age'] = league['Age_f'].fillna(league['Age'])
    league['FantasyTeam'] = league['FantasyTeam_f'].fillna('FA')
    league.drop(columns=['FPts_f', 'Age_f', 'FantasyTeam_f'], inplace=True)
    print(f"Fantrax merged: {league['FantasyTeam'].ne('FA').sum()} players matched")
except Exception as e:
    print(f"Fantrax merge skipped: {e}")

# 5. Rerun calc_value_score properly
def safe_row(r):
    return pd.Series({
        'PlayerName':    r.get('PlayerName', ''),
        'Age':           r.get('Age', 28),
        'FPts':          r.get('FPts', 0),
        'fantasy_pts':   r.get('fantasy_pts', 0),
        'savant_mult':   r.get('savant_mult', 1.0),
        'breakout_prob': r.get('breakout_prob', 0.0),
        'Pos':           r.get('Pos', ''),
        'PA':            r.get('PA', 500),
        'IP':            r.get('IP', 170),
        'availability':  r.get('availability', 0.85),
    })

league['value_score'] = league.apply(lambda r: calc_value_score(safe_row(r)), axis=1)

# 6. Rank
league = league.sort_values('value_score', ascending=False).reset_index(drop=True)
league['ModelRank']    = league.index + 1
league['MarketRank']   = league['ADP'].fillna(999)
league['TradeSurplus'] = league['MarketRank'] - league['ModelRank']

# 7. Safe HQ merge
# Removed 'MarketRank' and 'TradeSurplus' from the drop list
league.drop(columns=['HQ_Rank', 'HQ_Surplus', 'Name_clean'], errors='ignore', inplace=True)
league = league.drop_duplicates(subset=['PlayerName', 'player_type'], keep='first')
league['Name_clean'] = league['PlayerName'].apply(cleanname)
league = league.merge(top500_hq[['Name_clean', 'HQ_Rank']], on='Name_clean', how='left')
league['HQ_Rank']    = league['HQ_Rank'].fillna(999)
league['HQ_Surplus'] = league['HQ_Rank'] - league['ModelRank']

print("TOP 20 HQ Undervalued:")
print(league[league['HQ_Surplus'] > 20]
      .nlargest(20, 'HQ_Surplus')
      [['PlayerName', 'value_score', 'HQ_Rank', 'HQ_Surplus']]
      .to_string())

# 8. Fix Pos
league['Pos'] = league['Pos'].fillna('UTIL').astype(str).str.split('/').str[0]
league['Pos'] = league['Pos'].replace({'0.0': 'UTIL', 'nan': 'UTIL', 'P': 'SP', '': 'UTIL'})

Fantrax merged: 290 players matched
TOP 20 HQ Undervalued:
             PlayerName  value_score  HQ_Rank  HQ_Surplus
62         Leo De Vries   350.000000    999.0       936.0
123      Kazuma Okamoto   297.643739    999.0       875.0
168           Josh Bell   271.311781    999.0       830.0
199    Jake Cronenworth   258.948413    999.0       799.0
215      Jonathan India   246.461430    999.0       783.0
221       Nolan Arenado   241.960938    999.0       777.0
230        Willi Castro   237.962042    999.0       768.0
232          Joey Ortiz   235.707272    999.0       766.0
236        Nolan Gorman   233.001965    999.0       762.0
242       Brandon Marsh   229.712748    999.0       756.0
244        Ryan Jeffers   228.639720    999.0       754.0
245         Jeff McNeil   228.457863    999.0       753.0
254      Didier Fuentes   225.500000    999.0       744.0
255  George Lombard Jr.   225.500000    999.0       743.0
257         Trey Gibson   225.500000    999.0       741.0
258        Lu

In [ ]:
print(league.sort_values('value_score', ascending=False).head(11)[['PlayerName', 'Pos', 'value_score']].to_string())


               PlayerName Pos  value_score
0           Shohei Ohtani  DH   636.563077
1               Juan Soto  OF   563.622383
2          Bobby Witt Jr.  SS   559.988093
3             Aaron Judge  OF   557.675579
4             Paul Skenes  SP   546.712709
5            Tarik Skubal  SP   537.082873
6         Garrett Crochet  SP   516.561892
7        Gunnar Henderson  SS   513.209236
8   Vladimir Guerrero Jr.  1B   507.657270
9          Corbin Carroll  OF   492.221522
10             Nick Kurtz  1B   485.984702


# Player Clustering to find archetypes

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# First create cluster_score for master (separate from value_score)
master['cluster_score'] = master['fantasy_pts']  # Use fantasy_pts as base
# Add savant and breakout adjustments if they exist
if 'savant_mult' in master.columns:
    master['cluster_score'] *= master['savant_mult'].fillna(1.0)
if 'breakout_prob' in master.columns:
    master['cluster_score'] *= (1 + master['breakout_prob'] * 0.05)

hitter_stats = hitters[['xMLBAMID', 'HR', 'SB', 'BB', 'SO']].copy()
master = master.merge(hitter_stats, on='xMLBAMID', how='left', suffixes=('', '_hitter'))
master = master.drop(columns=['HR_hitter','SB_hitter','BB_hitter','SO_hitter'], errors='ignore')

features = ['HR', 'SB', 'BB', 'SO', 'savant_mult', 'breakout_prob']
cluster_data = master[features].fillna(0)

scaler = StandardScaler()
scaled = scaler.fit_transform(cluster_data)
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
master['cluster'] = kmeans.fit_predict(scaled)

# Archetypes
archetypes = []
for i in range(8):
    cluster_group = master[master['cluster'] == i]
    archetypes.append({
        'Cluster': i,
        'Top_Player': cluster_group.nlargest(1, 'cluster_score')['PlayerName'].iloc[0],
        'Avg_Value': cluster_group['cluster_score'].mean(),
        'ADP_Avg': cluster_group['ADP'].mean(),
        'Count': len(cluster_group)
    })

arche_df = pd.DataFrame(archetypes).sort_values('Avg_Value', ascending=False)
print("=== 8 PLAYER ARCHETYPES ===")
print(arche_df.round(1).to_string(index=False))


=== 8 PLAYER ARCHETYPES ===
 Cluster       Top_Player  Avg_Value  ADP_Avg  Count
       4    Shohei Ohtani      511.6     39.8     26
       2      Aaron Judge      409.4    187.0     86
       6        CJ Abrams      263.4    413.3     69
       5     Tarik Skubal      211.1    397.4     86
       1      Luis Arraez      187.9    722.6    155
       3   Framber Valdez       78.6    810.9    283
       7    Ernie Clement       60.4    944.6     92
       0 Brandon Woodruff        4.6    993.1   8546


# Player Breakdowns

In [ ]:
def player_breakdown(player_name):
    clean = cleanname(player_name)
    player_fantrax_data = fantrax[fantrax['Player'].apply(cleanname) == clean]
    player_master_data  = master[master['PlayerName'].apply(cleanname) == clean]

    if player_fantrax_data.empty or player_master_data.empty:
        print(f"❌ '{player_name}' not found — check spelling or try a partial match:")
        print(fantrax[fantrax['Player'].str.contains(player_name, case=False)]['Player'].tolist())
        return

    age         = player_fantrax_data['Age'].iloc[0]
    fpts        = player_fantrax_data['FPts'].iloc[0]
    fantasy_pts = player_master_data['fantasy_pts'].iloc[0]
    savant      = player_master_data.get('savant_mult', pd.Series([1.0])).iloc[0]
    team        = player_fantrax_data['FantasyTeam'].iloc[0]
    adp         = player_master_data['ADP'].iloc[0]
    age_mult    = dynasty_age_multiplier(age)

    # ── BUILD ROW ────────────────────────────────────────────────────────
    _mrow = master[master['PlayerName'].apply(cleanname) == clean]

    row = pd.Series({
        'PlayerName'   : player_name,
        'Age'          : age,
        'FPts'         : fpts,
        'fantasy_pts'  : fantasy_pts,
        'savant_mult'  : savant,
        'Pos'          : _mrow['Pos'].iloc[0] if not _mrow.empty else '',
        'breakout_prob': _mrow['breakout_prob'].iloc[0] if not _mrow.empty else 0.0,
        'availability' : float(_mrow['availability'].iloc[0]) if (not _mrow.empty and 'availability' in _mrow.columns) else 0.85,
        'PA'           : float(_mrow['PA'].iloc[0])  if (not _mrow.empty and 'PA'  in _mrow.columns and pd.notna(_mrow['PA'].iloc[0]))  else 500,
        'IP'           : float(_mrow['IP'].iloc[0])   if (not _mrow.empty and 'IP'  in _mrow.columns and pd.notna(_mrow['IP'].iloc[0]))  else 170
    })

    final = calc_value_score(row)

    # ── NOW EXTRACT AFTER ROW EXISTS ──────────────────────────────────────────
    availability = float(row.get('availability', 0.85))
    pos_val = row.get('Pos', '')
    if pos_val in ('SP', 'RP'):
      proj_pa = float(row.get('IP', 170))
    else:
      proj_pa = float(row.get('PA', 500))

    # ── BA RANK & BONUS ────────────────────────────────────────────────────────
    ba_rank = 999
    if 'ba_dynasty' in globals():
      ba_match = ba_dynasty[ba_dynasty['Nameclean'] == cleanname(player_name)]
      if not ba_match.empty:
        ba_rank = int(ba_match.iloc[0]['BARank'])
    rank_bonus_val = rank_bonus(ba_rank)

    # ── DISPLAY ───────────────────────────────────────────────────────────────
    prospect_match_disp = prospect_data[prospect_data['Name_clean'] == cleanname(player_name)]
    has_fv_disp  = not prospect_match_disp.empty
    prospect_fv_disp = int(prospect_match_disp.iloc[0]['FV']) if has_fv_disp else 40
    use_fv_disp = (
        (fpts < 100 and fantasy_pts < 50)
        or (has_fv_disp and prospect_fv_disp >= 50 and fpts < 300)
        or (cleanname(player_name) in {cleanname(p) for p in fv_override} and fpts < 300)
    ) and cleanname(player_name) not in {cleanname(p) for p in injury_override}
    score_method = "FV-based" if use_fv_disp else "Blended"

    print(f"{'─'*40}")
    print(f"  {player_name} ({team})")
    print(f"{'─'*40}")
    print(f"  Age:         {age}  →  age_mult: {age_mult:.2f}x")
    print(f"  FPts:        {fpts:.2f}  (2025 actual)")
    print(f"  fantasy_pts: {fantasy_pts:.2f}  (2026 projected)")
    print(f"  savant_mult: {savant:.4f}")
    print(f"  breakout_prob: {row['breakout_prob']:.4f}")
    print(f"  availability: {availability:.2f}x")
    print(f"  proj PA/IP:   {proj_pa:.0f}")
    print(f"  BA Rank:      {ba_rank}")
    print(f"  rank_bonus:   {rank_bonus(ba_rank):.0f}")
    print(f"  Scoring:     {score_method}")
    print(f"  ADP:         {adp:.1f}")
    print(f"{'─'*40}")
    print(f"  VALUE SCORE: {final:.2f}")
    print(f"{'─'*40}")


player_breakdown('George Lombard Jr.')
player_breakdown('Travis Sykora')
player_breakdown('Max Clark')
player_breakdown('Brice Turang')
player_breakdown('Matt Olson')
player_breakdown('Kris Bubic')

────────────────────────────────────────
  George Lombard Jr. (FA)
────────────────────────────────────────
  Age:         20  →  age_mult: 1.10x
  FPts:        0.00  (2025 actual)
  fantasy_pts: 0.45  (2026 projected)
  savant_mult: 1.0000
  breakout_prob: 0.0000
  availability: 0.85x
  proj PA/IP:   1
  BA Rank:      344
  rank_bonus:   1
  Scoring:     FV-based
  ADP:         999.0
────────────────────────────────────────
  VALUE SCORE: 225.50
────────────────────────────────────────
────────────────────────────────────────
  Travis Sykora (The Seamhead Express)
────────────────────────────────────────
  Age:         21  →  age_mult: 1.10x
  FPts:        0.00  (2025 actual)
  fantasy_pts: 1.16  (2026 projected)
  savant_mult: 1.0000
  breakout_prob: 0.0000
  availability: 0.85x
  proj PA/IP:   1
  BA Rank:      425
  rank_bonus:   0
  Scoring:     FV-based
  ADP:         999.0
────────────────────────────────────────
  VALUE SCORE: 199.10
────────────────────────────────────────
───

In [ ]:
player_breakdown('Jackson Chourio')  # Should show BA Rank ~50-100
player_breakdown('Paul Skenes')      # Top SP prospect


────────────────────────────────────────
  Jackson Chourio (Ashcrafts or Tatis)
────────────────────────────────────────
  Age:         22  →  age_mult: 1.10x
  FPts:        480.50  (2025 actual)
  fantasy_pts: 433.35  (2026 projected)
  savant_mult: 1.0410
  breakout_prob: 0.7303
  availability: 0.85x
  proj PA/IP:   594
  BA Rank:      12
  rank_bonus:   50
  Scoring:     Blended
  ADP:         19.4
────────────────────────────────────────
  VALUE SCORE: 434.96
────────────────────────────────────────
────────────────────────────────────────
  Paul Skenes (Harry Caray's Green Apples)
────────────────────────────────────────
  Age:         23  →  age_mult: 1.10x
  FPts:        452.75  (2025 actual)
  fantasy_pts: 401.79  (2026 projected)
  savant_mult: 1.0082
  breakout_prob: 0.9156
  availability: 0.85x
  proj PA/IP:   194
  BA Rank:      5
  rank_bonus:   50
  Scoring:     Blended
  ADP:         9.6
────────────────────────────────────────
  VALUE SCORE: 546.68
─────────────────────

In [ ]:
player_breakdown('Matt Olson')
player_breakdown('Brice Turang')
player_breakdown('Luis Gil')
player_breakdown('Shane Smith')
player_breakdown('Cam Smith')
player_breakdown('Nathan Eovaldi')
player_breakdown('Matt McLain')
player_breakdown('Agustin Ramirez')

────────────────────────────────────────
  Matt Olson (Hoos on First)
────────────────────────────────────────
  Age:         31  →  age_mult: 0.95x
  FPts:        506.00  (2025 actual)
  fantasy_pts: 455.79  (2026 projected)
  savant_mult: 1.1509
  breakout_prob: 0.9185
  availability: 0.85x
  proj PA/IP:   651
  BA Rank:      54
  rank_bonus:   25
  Scoring:     Blended
  ADP:         48.0
────────────────────────────────────────
  VALUE SCORE: 422.25
────────────────────────────────────────
────────────────────────────────────────
  Brice Turang (The Seamhead Express)
────────────────────────────────────────
  Age:         26  →  age_mult: 1.05x
  FPts:        445.50  (2025 actual)
  fantasy_pts: 398.30  (2026 projected)
  savant_mult: 0.9375
  breakout_prob: 0.0493
  availability: 0.85x
  proj PA/IP:   613
  BA Rank:      78
  rank_bonus:   25
  Scoring:     Blended
  ADP:         48.2
────────────────────────────────────────
  VALUE SCORE: 354.66
──────────────────────────────────

In [ ]:
names = [
    'Shohei Ohtani', 'Juan Soto', 'Bobby Witt Jr.', 'Vladimir Guerrero Jr.',
    'Aaron Judge', 'Gunnar Henderson', 'Corbin Carroll', 'Elly De La Cruz',
    'Fernando Tatis Jr.', 'Junior Caminero', 'Kyle Tucker', 'Nick Kurtz',
    'Jackson Chourio', 'Zach Neto', 'Kyle Schwarber', 'Cal Raleigh',
    'Yordan Alvarez', 'Pete Alonso', 'Wyatt Langford', 'Jackson Merrill'
]

check = league[league['PlayerName'].isin(names)][
    ['PlayerName','Age','FPts','fantasy_pts','savant_mult','breakout_prob','value_score']
].sort_values('value_score', ascending=False)

print(check.to_string(index=False))


           PlayerName  Age   FPts  fantasy_pts  savant_mult  breakout_prob  value_score
        Shohei Ohtani 31.0 852.44   594.629550     1.196354       0.825821   636.563077
            Juan Soto 27.0 645.50   556.647250     1.193142       0.936071   563.622383
       Bobby Witt Jr. 25.0 600.50   540.061550     1.171615       0.955716   559.988093
          Aaron Judge 33.0 677.00   561.124300     1.198524       0.852508   557.675579
     Gunnar Henderson 24.0 518.50   519.676650     1.143576       0.971803   513.209236
Vladimir Guerrero Jr. 27.0 533.50   527.110650     1.178646       0.982638   507.657270
       Corbin Carroll 25.0 531.00   508.394300     1.086545       0.883984   492.221522
           Nick Kurtz 23.0 504.50   455.568400     1.150174       0.697454   485.984702
   Fernando Tatis Jr. 27.0 518.50   503.253950     1.170833       0.985863   485.281504
      Junior Caminero 22.0 481.50   445.389150     1.152865       0.958616   474.620625
          Kyle Tucker 29.0 549.5

In [ ]:
league_export = league.copy()
league_export.index = range(1, len(league_export) + 1)
league_export.to_csv('seamhead_league_values_2026.csv', index_label='Rank')

In [ ]:
print(league.sort_values('value_score', ascending=False).head(11)[['PlayerName', 'Pos', 'value_score']].to_string())


               PlayerName Pos  value_score
0           Shohei Ohtani  DH   636.563077
1               Juan Soto  OF   563.622383
2          Bobby Witt Jr.  SS   559.988093
3             Aaron Judge  OF   557.675579
4             Paul Skenes  SP   546.712709
5            Tarik Skubal  SP   537.082873
6         Garrett Crochet  SP   516.561892
7        Gunnar Henderson  SS   513.209236
8   Vladimir Guerrero Jr.  1B   507.657270
9          Corbin Carroll  OF   492.221522
10             Nick Kurtz  1B   485.984702


In [ ]:
top_league = league.sort_values('value_score', ascending=False).head(20)
for name in top_league['PlayerName']:
  player_breakdown(name)

────────────────────────────────────────
  Shohei Ohtani (Baltimore KnOrioles)
────────────────────────────────────────
  Age:         31  →  age_mult: 0.95x
  FPts:        852.44  (2025 actual)
  fantasy_pts: 594.63  (2026 projected)
  savant_mult: 1.1964
  breakout_prob: 0.8225
  availability: 0.85x
  proj PA/IP:   658
  BA Rank:      1
  rank_bonus:   50
  Scoring:     Blended
  ADP:         1.3
────────────────────────────────────────
  VALUE SCORE: 636.52
────────────────────────────────────────
────────────────────────────────────────
  Juan Soto (Spinmaster Wizards)
────────────────────────────────────────
  Age:         27  →  age_mult: 1.00x
  FPts:        645.50  (2025 actual)
  fantasy_pts: 556.65  (2026 projected)
  savant_mult: 1.1931
  breakout_prob: 0.9361
  availability: 0.85x
  proj PA/IP:   621
  BA Rank:      2
  rank_bonus:   50
  Scoring:     Blended
  ADP:         4.7
────────────────────────────────────────
  VALUE SCORE: 563.62
──────────────────────────────────

In [ ]:
# ── KEEPERS ───────────────────────────────────────────────────
print("=== KEEPERS (Top 22) ===")
for player in my_roster[my_roster['KEEP'] == True]['PlayerName']:
    player_breakdown(player)

# ── ON THE BUBBLE ─────────────────────────────────────────────
print("\n=== ON THE BUBBLE (23–27) ===")
for player in my_roster[my_roster['KEEP'] == False]['PlayerName'].head(5):
    player_breakdown(player)

=== KEEPERS (Top 22) ===
────────────────────────────────────────
  Aaron Judge (The Seamhead Express)
────────────────────────────────────────
  Age:         33  →  age_mult: 0.95x
  FPts:        677.00  (2025 actual)
  fantasy_pts: 561.12  (2026 projected)
  savant_mult: 1.1985
  breakout_prob: 0.8525
  availability: 0.85x
  proj PA/IP:   633
  BA Rank:      10
  rank_bonus:   50
  Scoring:     Blended
  ADP:         1.9
────────────────────────────────────────
  VALUE SCORE: 557.68
────────────────────────────────────────
────────────────────────────────────────
  Freddie Freeman (The Seamhead Express)
────────────────────────────────────────
  Age:         36  →  age_mult: 0.85x
  FPts:        453.00  (2025 actual)
  fantasy_pts: 478.64  (2026 projected)
  savant_mult: 1.1177
  breakout_prob: 0.9646
  availability: 0.85x
  proj PA/IP:   658
  BA Rank:      68
  rank_bonus:   25
  Scoring:     Blended
  ADP:         67.9
────────────────────────────────────────
  VALUE SCORE: 415.00

## Trade Surplus

In [ ]:
my_roster['value_score'] = my_roster.apply(calc_value_score, axis=1)
my_roster = my_roster.sort_values('value_score', ascending=False).reset_index(drop=True)
my_roster.index += 1
my_roster['KEEP'] = my_roster.index <= 22

print(my_roster[['PlayerName', 'Position', 'Age', 'FPts', 'fantasy_pts', 'value_score', 'KEEP']].to_string())
my_roster.to_csv('/content/drive/MyDrive/WeLoveSabermetrics/seamhead_keeper_decisions.csv', index_label='Rank')
print("\n✅ Saved to Drive")


         PlayerName Position  Age    FPts  fantasy_pts  value_score  KEEP
1       Aaron Judge       OF   33  677.00   561.124300   557.604229  True
2   Freddie Freeman       1B   36  453.00   478.639700   415.016603  True
3         Max Clark       OF   21   47.50     1.289350   357.500000  True
4      Brice Turang       2B   26  445.50   398.295500   354.662782  True
5          Ian Happ       OF   31  436.50   397.765700   348.748587  True
6          Ben Rice     C,1B   27  394.50   331.412000   337.960032  True
7    Bryan Reynolds       OF   31  402.50   365.731500   331.517464  True
8      Matt Chapman       3B   32  381.50   381.594250   330.693512  True
9   Shea Langeliers        C   28  381.50   348.183350   320.792577  True
10  Brendan Donovan       2B   29  340.00   343.136600   305.427716  True
11   Freddy Peralta       SP   29  309.25   262.394825   304.620622  True
12  Xander Bogaerts       SS   33  382.00   334.076900   275.832002  True
13      Sal Stewart       1B   22  305

In [ ]:
# Trade-focused surplus view for your roster
trade_board = my_roster[['PlayerName', 'value_score', 'ADP']].copy()
trade_board = trade_board.sort_values('value_score', ascending=False).reset_index(drop=True)
trade_board.index += 1  # 1-based rank

trade_board['ModelRank'] = trade_board.index
trade_board['MarketRank'] = trade_board['ADP']
trade_board['TradeSurplus'] = trade_board['MarketRank'] - trade_board['ModelRank']

print("=== TRADE SURPLUS BOARD (your team only) ===")
print(trade_board[['PlayerName', 'value_score', 'MarketRank', 'TradeSurplus']].head(30).to_string())


=== TRADE SURPLUS BOARD (your team only) ===
         PlayerName  value_score  MarketRank  TradeSurplus
1       Aaron Judge   557.604229         1.9           0.9
2   Freddie Freeman   415.016603        54.6          52.6
3         Max Clark   357.500000       478.2         475.2
4      Brice Turang   354.662782        39.9          35.9
5          Ian Happ   348.748587       113.4         108.4
6          Ben Rice   337.960032        76.2          70.2
7    Bryan Reynolds   331.517464       163.0         156.0
8      Matt Chapman   330.693512       112.7         104.7
9   Shea Langeliers   320.792577        70.8          61.8
10  Brendan Donovan   305.427716       237.2         227.2
11   Freddy Peralta   304.620622        57.1          46.1
12  Xander Bogaerts   275.832002       313.9         301.9
13      Sal Stewart   273.179221       215.1         202.1
14     Tanner Bibee   255.713405       160.8         146.8
15     Matt Wallner   234.221422       299.4         284.4
16  Elmer R

In [ ]:
# ── MODEL CONVICTION PLAYS (your model > HQ > market) ────────────────────────
model_conviction = league[
    (league['HQ_Rank'] < 999) &
    (league['HQ_Surplus'] > 10) &       # Relaxed from > 20
    (league['TradeSurplus'] > 5) &     # Relaxed from > 10
    (league['value_score'] > 300) &
    (league['FantasyTeam'] != 'The Seamhead Express')
].sort_values('HQ_Surplus', ascending=False)

print("=== MODEL CONVICTION PLAYS (you > HQ > market) ===")
print(model_conviction[['PlayerName','FantasyTeam','value_score','HQ_Rank','HQ_Surplus','TradeSurplus']].head(20).to_string(index=False))

# ── HQ BUY LOW (HQ loves > you > market) ─────────────────────────────────────
hq_buy_low = league[
    (league['HQ_Rank'] < 999) &
    (league['HQ_Surplus'] < -10) &      # Relaxed from < -20
    (league['TradeSurplus'] > 5) &     # Relaxed from > 10
    (league['value_score'] > 300) &
    (league['FantasyTeam'] != 'The Seamhead Express')
].sort_values('HQ_Surplus')

print("\n=== HQ BUY LOW (HQ loves > you > market) ===")
print(hq_buy_low[['PlayerName','FantasyTeam','value_score','HQ_Rank','HQ_Surplus','TradeSurplus']].head(20).to_string(index=False))

=== MODEL CONVICTION PLAYS (you > HQ > market) ===
       PlayerName             FantasyTeam  value_score  HQ_Rank  HQ_Surplus  TradeSurplus
      Jake Burger                      FA   301.028244    442.0       323.0    117.720001
      Luis Arraez           Hoos on First   322.810473    386.0       289.0    183.049988
   Gleyber Torres Madoff's Biggest Client   349.040111    337.0       272.0    196.510010
   Nolan Schanuel               Markyball   313.704337    339.0       236.0    491.659973
    Alec Burleson   $5B Dollar Shave Club   343.235614    308.0       233.0    100.190002
Munetaka Murakami                      FA   343.808174    291.0       217.0    925.000000
     Jakob Marsee            morning WOOD   339.721463    284.0       208.0     61.080002
    Brandon Nimmo     Baltimore KnOrioles   365.159650    254.0       204.0     95.460007
       Masyn Winn           Hoos on First   337.445556    280.0       199.0    283.779999
   Ezequiel Tovar    The Boston Red Shawx   321.1

In [ ]:
player_breakdown('Ryan Weathers')
player_breakdown('Will Warren')
player_breakdown('Gerrit Cole')
player_breakdown('Kris Bubic')
player_breakdown('Brendan Donovan')


────────────────────────────────────────
  Ryan Weathers (Hoos on First)
────────────────────────────────────────
  Age:         26  →  age_mult: 1.05x
  FPts:        147.75  (2025 actual)
  fantasy_pts: 163.12  (2026 projected)
  savant_mult: 1.0386
  breakout_prob: 0.0133
  availability: 0.85x
  proj PA/IP:   124
  BA Rank:      384
  rank_bonus:   0
  Scoring:     Blended
  ADP:         232.6
────────────────────────────────────────
  VALUE SCORE: 149.99
────────────────────────────────────────
────────────────────────────────────────
  Will Warren (FA)
────────────────────────────────────────
  Age:         26  →  age_mult: 1.05x
  FPts:        111.50  (2025 actual)
  fantasy_pts: 160.63  (2026 projected)
  savant_mult: 1.1482
  breakout_prob: 0.0200
  availability: 0.85x
  proj PA/IP:   134
  BA Rank:      292
  rank_bonus:   1
  Scoring:     Blended
  ADP:         285.0
────────────────────────────────────────
  VALUE SCORE: 177.30
────────────────────────────────────────
───────

# Trade Grader

In [ ]:
def trade_grader(team_a_gives, team_b_gives, threshold=0.10):
  '''
    team_a_gives : list of player name strings Team A is sending
    team_b_gives : list of player name strings Team B is receiving
    threshold    : % difference allowed before flagging as uneven (default 10%)
  '''

  def get_value(player_name):
    clean = cleanname(player_name)
    match = league[league['PlayerName'].apply(cleanname) == clean]
    if match.empty:
      print(f"  ⚠️  '{player_name}' not found in league — skipping")
      return 0.0, '???', '???'
    row = match.iloc[0]
    return row['value_score'], row.get('FantasyTeam', 'FA'), row.get('Pos', '?')

  print(f"{'═'*50}")
  print(f"  📊 TRADE GRADER")
  print(f"{'═'*50}")

  # ── SIDE A ────────────────────────────────────────
  print(f"\n  SIDE A GIVES:")
  side_a_total = 0.0
  for p in team_a_gives:
    val, team, pos = get_value(p)
    side_a_total += val
    print(f"    {p:<25} ({pos})  →  {val:.1f}")

  # ── SIDE B ────────────────────────────────────────
  print(f"\n  SIDE B GIVES:")
  side_b_total = 0.0
  for p in team_b_gives:
    val, team, pos = get_value(p)
    side_b_total += val
    print(f"    {p:<25} ({pos})  →  {val:.1f}")

  # ── VERDICT ───────────────────────────────────────
  diff     = abs(side_a_total - side_b_total)
  pct_diff = diff / max(side_a_total, side_b_total) if max(side_a_total, side_b_total) > 0 else 0.0
  winner   = 'SIDE A' if side_b_total > side_a_total else 'SIDE B'
  surplus  = abs(side_b_total - side_a_total)

  print(f"\n{'─'*50}")
  print(f"  Side A total:  {side_a_total:.1f}")
  print(f"  Side B total:  {side_b_total:.1f}")
  print(f"  Difference:    {surplus:.1f}  ({pct_diff:.1%})")
  print(f"{'─'*50}")

  if pct_diff <= threshold:
    print(f"  ✅  FAIR TRADE  —  within {threshold:.0%} value threshold")
  else:
    print(f"  ❌  UNEVEN  —  {winner} gets the better end by {surplus:.1f} pts ({pct_diff:.1%})")
  print(f"{'='*50}\n")